In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/processed/saas_customers_cleaned.csv",
                  parse_dates=["Subscription_Date", "Renewal_Date", "Cancellation_Date"])
df["Revenue"] = df["Monthly_Fee"] * (1 - df["Discount_Percent"] / 100)

conn = sqlite3.connect("../data/saas_analytics.db")
df.to_sql("customers", conn, if_exists="replace", index=False)

print("Table loaded. Row count check:")
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM customers", conn)
print(result)


In [ ]:
# Q1: List all Enterprise plan customers
q1 = """
SELECT Customer_ID, Country, Monthly_Fee, Customer_Status
FROM customers
WHERE Subscription_Plan = 'Enterprise'
LIMIT 10;
"""
print(pd.read_sql(q1, conn))

In [ ]:
# Q2: Find all churned customers with high monthly fees (over $100)
q2 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee, Customer_Satisfaction
FROM customers
WHERE Customer_Status = 'Churned' AND Monthly_Fee > 100
LIMIT 10;
"""
print(pd.read_sql(q2, conn))

In [ ]:
# Q3: Customers from USA or India using the Premium plan
q3 = """
SELECT Customer_ID, Country, Subscription_Plan, Monthly_Fee
FROM customers
WHERE Country IN ('USA', 'India') AND Subscription_Plan = 'Premium'
LIMIT 10;
"""
print(pd.read_sql(q3, conn))

In [ ]:
# Q4: Customers who never filed a support ticket
q4 = """
SELECT Customer_ID, Subscription_Plan, Usage_Hours_Monthly
FROM customers
WHERE Support_Tickets = 0
LIMIT 10;
"""
print(pd.read_sql(q4, conn))

In [ ]:
# Q5: Customers with satisfaction below 4 (at-risk / unhappy customers)
q5 = """
SELECT Customer_ID, Customer_Satisfaction, Support_Tickets, Customer_Status
FROM customers
WHERE Customer_Satisfaction < 4
ORDER BY Customer_Satisfaction ASC
LIMIT 10;
"""
print(pd.read_sql(q5, conn))

In [ ]:
# Q6: Count of customers per subscription plan
q6 = """
SELECT Subscription_Plan, COUNT(*) as Customer_Count
FROM customers
GROUP BY Subscription_Plan
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q6, conn))

In [ ]:
# Q7: Average monthly fee and average satisfaction per plan
q7 = """
SELECT Subscription_Plan,
       ROUND(AVG(Monthly_Fee), 2) as Avg_Fee,
       ROUND(AVG(Customer_Satisfaction), 2) as Avg_Satisfaction
FROM customers
GROUP BY Subscription_Plan
ORDER BY Avg_Fee DESC;
"""
print(pd.read_sql(q7, conn))

In [ ]:
# Q8: Countries with more than 3000 customers (using HAVING)
q8 = """
SELECT Country, COUNT(*) as Customer_Count
FROM customers
GROUP BY Country
HAVING COUNT(*) > 3000
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q8, conn))

In [ ]:
# Q9: Churn rate by marketing channel
q9 = """
SELECT Marketing_Channel,
       COUNT(*) as Total_Customers,
       SUM(CASE WHEN Customer_Status = 'Churned' THEN 1 ELSE 0 END) as Churned_Count,
       ROUND(100.0 * SUM(CASE WHEN Customer_Status = 'Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate_Pct
FROM customers
GROUP BY Marketing_Channel
ORDER BY Churn_Rate_Pct DESC;
"""
print(pd.read_sql(q9, conn))

In [ ]:
# Q10: Average lifetime value by company size
q10 = """
SELECT Company_Size,
       COUNT(*) as Customer_Count,
       ROUND(AVG(Lifetime_Value), 2) as Avg_LTV
FROM customers
GROUP BY Company_Size
ORDER BY Avg_LTV DESC;
"""
print(pd.read_sql(q10, conn))

In [ ]:
# Q11: Categorize customers into value tiers based on Lifetime Value
q11 = """
SELECT Customer_ID, Lifetime_Value,
    CASE
        WHEN Lifetime_Value >= 3000 THEN 'High Value'
        WHEN Lifetime_Value >= 500 THEN 'Medium Value'
        ELSE 'Low Value'
    END as Value_Tier
FROM customers
ORDER BY Lifetime_Value DESC
LIMIT 10;
"""
print(pd.read_sql(q11, conn))

In [ ]:
# Q12: Count customers in each value tier
q12 = """
SELECT
    CASE
        WHEN Lifetime_Value >= 3000 THEN 'High Value'
        WHEN Lifetime_Value >= 500 THEN 'Medium Value'
        ELSE 'Low Value'
    END as Value_Tier,
    COUNT(*) as Customer_Count,
    ROUND(AVG(Lifetime_Value), 2) as Avg_LTV_In_Tier
FROM customers
GROUP BY Value_Tier
ORDER BY Avg_LTV_In_Tier DESC;
"""
print(pd.read_sql(q12, conn))

In [ ]:
# Q13: Top 10 customers by revenue, with a risk flag for high support tickets
q13 = """
SELECT Customer_ID, Monthly_Fee, Support_Tickets,
    CASE WHEN Support_Tickets >= 5 THEN 'At Risk' ELSE 'Healthy' END as Risk_Flag
FROM customers
WHERE Customer_Status = 'Active'
ORDER BY Monthly_Fee DESC
LIMIT 10;
"""
print(pd.read_sql(q13, conn))

In [ ]:
# Q14: Age groups and their average satisfaction
q14 = """
SELECT
    CASE
        WHEN Age < 30 THEN 'Under 30'
        WHEN Age < 45 THEN '30-44'
        WHEN Age < 60 THEN '45-59'
        ELSE '60+'
    END as Age_Group,
    COUNT(*) as Customer_Count,
    ROUND(AVG(Customer_Satisfaction), 2) as Avg_Satisfaction
FROM customers
GROUP BY Age_Group
ORDER BY Age_Group;
"""
print(pd.read_sql(q14, conn))

In [ ]:
# Q15: Sort customers by discount given, highest first, showing plan and fee
q15 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee, Discount_Percent
FROM customers
ORDER BY Discount_Percent DESC
LIMIT 10;
"""
print(pd.read_sql(q15, conn))

In [ ]:
plan_details = pd.DataFrame({
    "Subscription_Plan": ["Basic", "Standard", "Premium", "Enterprise"],
    "Max_Users_Allowed": [5, 20, 100, 1000],
    "Support_Level": ["Email Only", "Email + Chat", "Priority Support", "Dedicated Account Manager"],
    "Onboarding_Included": [0, 0, 1, 1]  # 0 = No, 1 = Yes
})

plan_details.to_sql("plan_details", conn, if_exists="replace", index=False)
print(pd.read_sql("SELECT * FROM plan_details", conn))

In [ ]:
# Q16: Show customers along with their plan's support level and max users
q16 = """
SELECT c.Customer_ID, c.Subscription_Plan, c.Monthly_Fee,
       p.Support_Level, p.Max_Users_Allowed
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
LIMIT 10;
"""
print(pd.read_sql(q16, conn))

In [ ]:
# Q17: Count how many customers get onboarding included, broken down by plan
q17 = """
SELECT p.Subscription_Plan, p.Onboarding_Included, COUNT(*) as Customer_Count
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
GROUP BY p.Subscription_Plan, p.Onboarding_Included
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q17, conn))

In [ ]:
# Q18: Average satisfaction for customers who get Priority Support vs not
q18 = """
SELECT p.Support_Level, ROUND(AVG(c.Customer_Satisfaction), 2) as Avg_Satisfaction, COUNT(*) as Customer_Count
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
GROUP BY p.Support_Level
ORDER BY Avg_Satisfaction DESC;
"""
print(pd.read_sql(q18, conn))

In [ ]:
# Q19: Enterprise customers exceeding their plan's max users would be flagged here
# (illustrative - our dataset doesn't track actual user count per account, 
#  but this shows how you'd structure the check if it did)
q19 = """
SELECT c.Customer_ID, c.Subscription_Plan, p.Max_Users_Allowed
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
WHERE c.Subscription_Plan = 'Enterprise'
LIMIT 10;
"""
print(pd.read_sql(q19, conn))

In [ ]:
# Q20: LEFT JOIN example - show all plans, even if (hypothetically) some had zero customers
q20 = """
SELECT p.Subscription_Plan, p.Support_Level, COUNT(c.Customer_ID) as Customer_Count
FROM plan_details p
LEFT JOIN customers c ON p.Subscription_Plan = c.Subscription_Plan
GROUP BY p.Subscription_Plan, p.Support_Level
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q20, conn))

In [ ]:
# Q21: Customers whose Monthly_Fee is above the overall average
q21 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee
FROM customers
WHERE Monthly_Fee > (SELECT AVG(Monthly_Fee) FROM customers)
ORDER BY Monthly_Fee DESC
LIMIT 10;
"""
print(pd.read_sql(q21, conn))

In [ ]:
# Q22: Countries with above-average churn rate
q22 = """
SELECT Country,
       ROUND(100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate_Pct
FROM customers
GROUP BY Country
HAVING Churn_Rate_Pct > (
    SELECT 100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*)
    FROM customers
)
ORDER BY Churn_Rate_Pct DESC;
"""
print(pd.read_sql(q22, conn))

In [ ]:
# Q23 (rewritten with a window function - much faster)
q23 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee, Plan_Avg_Fee
FROM (
    SELECT Customer_ID, Subscription_Plan, Monthly_Fee,
           AVG(Monthly_Fee) OVER (PARTITION BY Subscription_Plan) as Plan_Avg_Fee
    FROM customers
)
WHERE Monthly_Fee > Plan_Avg_Fee
ORDER BY Monthly_Fee DESC
LIMIT 10;
"""
print(pd.read_sql(q23, conn))

In [ ]:
# Q24: The single most valuable customer (highest Lifetime_Value)
q24 = """
SELECT Customer_ID, Subscription_Plan, Lifetime_Value
FROM customers
WHERE Lifetime_Value = (SELECT MAX(Lifetime_Value) FROM customers);
"""
print(pd.read_sql(q24, conn))

In [ ]:
# Q25: Customers with fewer support tickets than the average for churned customers
q25 = """
SELECT Customer_ID, Customer_Status, Support_Tickets
FROM customers
WHERE Support_Tickets < (
    SELECT AVG(Support_Tickets) FROM customers WHERE Customer_Status = 'Churned'
)
AND Customer_Status = 'Active'
LIMIT 10;
"""
print(pd.read_sql(q25, conn))

In [ ]:
# Q26: Using a CTE to first calculate revenue per plan, then filter
q26 = """
WITH plan_revenue AS (
    SELECT Subscription_Plan, SUM(Revenue) as Total_Revenue, COUNT(*) as Customer_Count
    FROM customers
    WHERE Customer_Status = 'Active'
    GROUP BY Subscription_Plan
)
SELECT * FROM plan_revenue
ORDER BY Total_Revenue DESC;
"""
print(pd.read_sql(q26, conn))

In [ ]:
# Q27: CTE + JOIN - combine plan revenue with plan metadata
q27 = """
WITH plan_revenue AS (
    SELECT Subscription_Plan, SUM(Revenue) as Total_Revenue
    FROM customers
    WHERE Customer_Status = 'Active'
    GROUP BY Subscription_Plan
)
SELECT pr.Subscription_Plan, pr.Total_Revenue, pd.Support_Level
FROM plan_revenue pr
JOIN plan_details pd ON pr.Subscription_Plan = pd.Subscription_Plan
ORDER BY pr.Total_Revenue DESC;
"""
print(pd.read_sql(q27, conn))

In [ ]:
# Q28: Multiple CTEs - compare high-value vs low-value customer segments
q28 = """
WITH high_value AS (
    SELECT COUNT(*) as Count, AVG(Customer_Satisfaction) as Avg_Satisfaction
    FROM customers WHERE Lifetime_Value >= 2000
),
low_value AS (
    SELECT COUNT(*) as Count, AVG(Customer_Satisfaction) as Avg_Satisfaction
    FROM customers WHERE Lifetime_Value < 2000
)
SELECT 'High Value' as Segment, Count, ROUND(Avg_Satisfaction,2) as Avg_Satisfaction FROM high_value
UNION ALL
SELECT 'Low Value' as Segment, Count, ROUND(Avg_Satisfaction,2) as Avg_Satisfaction FROM low_value;
"""
print(pd.read_sql(q28, conn))

In [ ]:
# Q29: CTE to find each country's top-paying customer
q29 = """
WITH ranked_customers AS (
    SELECT Customer_ID, Country, Monthly_Fee,
           ROW_NUMBER() OVER (PARTITION BY Country ORDER BY Monthly_Fee DESC) as rn
    FROM customers
)
SELECT Country, Customer_ID, Monthly_Fee
FROM ranked_customers
WHERE rn = 1
ORDER BY Monthly_Fee DESC;
"""
print(pd.read_sql(q29, conn))

In [ ]:
# Q30: CTE calculating churn rate by industry, filtered to only industries with >1000 customers
q30 = """
WITH industry_stats AS (
    SELECT Industry,
           COUNT(*) as Total,
           SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) as Churned,
           ROUND(100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate
    FROM customers
    GROUP BY Industry
)
SELECT * FROM industry_stats
WHERE Total > 1000
ORDER BY Churn_Rate DESC;
"""
print(pd.read_sql(q30, conn))

In [ ]:
# Q31: Rank customers by Lifetime_Value within their own plan
q31 = """
SELECT Customer_ID, Subscription_Plan, Lifetime_Value,
       RANK() OVER (PARTITION BY Subscription_Plan ORDER BY Lifetime_Value DESC) as Rank_In_Plan
FROM customers
WHERE Customer_Status = 'Active'
ORDER BY Subscription_Plan, Rank_In_Plan
LIMIT 15;
"""
print(pd.read_sql(q31, conn))

In [ ]:
# Q32: DENSE_RANK - like RANK but no gaps after ties
q32 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee,
       DENSE_RANK() OVER (ORDER BY Monthly_Fee DESC) as Fee_Rank
FROM customers
LIMIT 10;
"""
print(pd.read_sql(q32, conn))

In [ ]:
# Q33: Running total of revenue, ordered by subscription date (cumulative growth)
q33 = """
SELECT Customer_ID, Subscription_Date, Revenue,
       SUM(Revenue) OVER (ORDER BY Subscription_Date) as Running_Total_Revenue
FROM customers
WHERE Customer_Status = 'Active'
ORDER BY Subscription_Date
LIMIT 15;
"""
print(pd.read_sql(q33, conn))

In [ ]:
# Q34: LAG - compare each customer's fee to the previous customer's fee (ordered by signup)
q34 = """
SELECT Customer_ID, Subscription_Date, Monthly_Fee,
       LAG(Monthly_Fee) OVER (ORDER BY Subscription_Date) as Previous_Customer_Fee
FROM customers
ORDER BY Subscription_Date
LIMIT 10;
"""
print(pd.read_sql(q34, conn))

In [ ]:
# Q35: NTILE - split customers into 4 quartiles by Lifetime_Value
q35 = """
SELECT Customer_ID, Lifetime_Value,
       NTILE(4) OVER (ORDER BY Lifetime_Value DESC) as Value_Quartile
FROM customers
LIMIT 20;
"""
print(pd.read_sql(q35, conn))

In [ ]:
# Q36: Percentage of total revenue each plan contributes (window function version)
q36 = """
SELECT Subscription_Plan,
       SUM(Revenue) as Plan_Revenue,
       ROUND(100.0 * SUM(Revenue) / SUM(SUM(Revenue)) OVER (), 2) as Pct_Of_Total_Revenue
FROM customers
WHERE Customer_Status = 'Active'
GROUP BY Subscription_Plan
ORDER BY Pct_Of_Total_Revenue DESC;
"""
print(pd.read_sql(q36, conn))

In [ ]:
# Q37: Month-over-month change in new signups (using LAG)
q37 = """
WITH monthly_signups AS (
    SELECT strftime('%Y-%m', Subscription_Date) as Month, COUNT(*) as Signups
    FROM customers
    GROUP BY Month
)
SELECT Month, Signups,
       Signups - LAG(Signups) OVER (ORDER BY Month) as Change_From_Prev_Month
FROM monthly_signups
ORDER BY Month
LIMIT 15;
"""
print(pd.read_sql(q37, conn))

In [ ]:
# Q38: Top 3 customers by Lifetime_Value per country
q38 = """
WITH ranked AS (
    SELECT Customer_ID, Country, Lifetime_Value,
           ROW_NUMBER() OVER (PARTITION BY Country ORDER BY Lifetime_Value DESC) as rn
    FROM customers
)
SELECT Country, Customer_ID, Lifetime_Value
FROM ranked
WHERE rn <= 3
ORDER BY Country, rn;
"""
print(pd.read_sql(q38, conn))

In [ ]:
# Q39: Cumulative percentage of customers reached, ordered by Lifetime_Value (useful for Pareto/80-20 analysis)
q39 = """
WITH ranked AS (
    SELECT Customer_ID, Lifetime_Value,
           ROW_NUMBER() OVER (ORDER BY Lifetime_Value DESC) as rn,
           COUNT(*) OVER () as total_customers
    FROM customers
)
SELECT Customer_ID, Lifetime_Value, rn,
       ROUND(100.0 * rn / total_customers, 2) as Cumulative_Pct_Of_Customers
FROM ranked
LIMIT 10;
"""
print(pd.read_sql(q39, conn))

In [ ]:
# Q40: Final summary query - full KPI snapshot in one query
q40 = """
SELECT
    COUNT(*) as Total_Customers,
    SUM(CASE WHEN Customer_Status='Active' THEN 1 ELSE 0 END) as Active_Customers,
    SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) as Churned_Customers,
    ROUND(100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate_Pct,
    ROUND(SUM(CASE WHEN Customer_Status='Active' THEN Revenue ELSE 0 END), 2) as MRR,
    ROUND(AVG(CASE WHEN Customer_Status='Active' THEN Revenue END), 2) as ARPU,
    ROUND(AVG(Lifetime_Value), 2) as Avg_CLTV
FROM customers;
"""
print(pd.read_sql(q40, conn))

In [ ]:
queries = {
    "01_10_basic_select_groupby": [q1, q2, q3, q4, q5, q6, q7, q8, q9, q10],
    "11_20_case_joins": [q11, q12, q13, q14, q15, q16, q17, q18, q19, q20],
    "21_30_subqueries_ctes": [q21, q22, q23, q24, q25, q26, q27, q28, q29, q30],
    "31_40_window_functions": [q31, q32, q33, q34, q35, q36, q37, q38, q39, q40],
}

for filename, query_list in queries.items():
    with open(f"../sql/{filename}.sql", "w") as f:
        for i, q in enumerate(query_list, 1):
            f.write(f"-- Query {i}\n{q.strip()}\n\n")

print("All SQL files saved to sql/ folder")